# Generalization analysis
Ten-fold outer and five-fold inner nested cross-validation. Hyperparameters and training epochs, k-means centroids, and cluster-to-subtype mappings are determined without using the held-out outer fold.

In [ ]:
from pathlib import Path
import json
import sys
import torch

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_DIR))
from src import config
from src.generalization import aggregate_outer_folds, analysis_configuration, run_outer_fold

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RESULT_DIR = config.OUTPUT_DIR / 'nested_cv'
print('Device:', DEVICE)
print('Result directory:', RESULT_DIR)

In [ ]:
processed = torch.load(config.PROCESSED_DATA_PATH, map_location='cpu', weights_only=False)
manifest = processed['participant_manifest']
assert len(manifest) == len(processed['input_genotype'])
print('Participants:', len(manifest))
print('Outer-fold counts:', manifest['outer_fold'].value_counts().sort_index().to_dict())

In [ ]:
RESULT_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULT_DIR / 'analysis_configuration.json', 'w') as handle:
    json.dump(analysis_configuration(config), handle, indent=2)
for outer_fold in range(1, config.OUTER_SPLITS + 1):
    row, _, _ = run_outer_fold(processed, outer_fold, config, DEVICE, RESULT_DIR)
    print(f"[Completed] Outer fold {outer_fold}/{config.OUTER_SPLITS} | ACC={row['accuracy']:.4f} | ARI={row['ari']:.4f}", flush=True)
metrics, class_metrics, predictions = aggregate_outer_folds(RESULT_DIR)

In [ ]:
metrics_path = RESULT_DIR / 'outer_fold_metrics.csv'
if metrics_path.exists():
    import pandas as pd
    metrics = pd.read_csv(metrics_path)
    display(metrics[['accuracy','balanced_accuracy','ari','macro_precision','macro_recall','macro_f1']].agg(['mean','std']).T)
else:
    print('No completed aggregate result found.')